# **Laboratorium 4 - ETL w hurtowniach danych**
## Zadanie 1 - Budowa pipeline ETL dla danych sprzedaży

Celem zadania jest wykonanie procesu ETL:
- Extract – wczytanie danych
- Transform – czyszczenie i przygotowanie danych
- Load – zapis danych do tabeli faktów

## 1. Extract - wczytanie danych

Wczytujemy dane do Pandas oraz analizujemy ich strukturę.

In [1]:
import pandas as pd

df = pd.read_csv(
    "Online_Retail.csv",
    encoding="ISO-8859-1",
    sep=",",
    on_bad_lines="skip"
)

df.head()
df.info()
df.describe()

/tmp/ipykernel_6804/3063170307.py:3: DtypeWarning: Columns (3,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 784289 entries, 0 to 784288
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   InvoiceNo    784289 non-null  object
 1   StockCode    784289 non-null  object
 2   Description  782481 non-null  object
 3   Quantity     784283 non-null  object
 4   InvoiceDate  784284 non-null  object
 5   UnitPrice    784278 non-null  object
 6   CustomerID   598207 non-null  object
 7   Country      784273 non-null  object
dtypes: object(8)
memory usage: 47.9+ MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,784289,784289,782481,784283.0,784284,784278.00,598207.0,784273
unique,25900,4076,4235,1107.0,23268,2107.00,6615.0,40
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,1.0,10/31/11 14:41,1.25,17841.0,United Kingdom
freq,2228,3075,3151,177645.0,2228,54752.00,10174.0,716791


## 2. Transform - czyszczenie danych

W tym kroku:
- usuwamy brakujące CustomerID
- usuwamy błędne wartości (Quantity <= 0, UnitPrice < 0)
- usuwamy duplikaty
- poprawiamy typy danych

In [2]:
# konwersja na liczby
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# usunięcie braków
df = df.dropna(subset=["CustomerID"])

# usunięcie błędnych danych
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] >= 0]

# usunięcie duplikatów
df = df.drop_duplicates()

# konwersja typów
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")

df.head()

/tmp/ipykernel_6804/1531349075.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 3. Transform - przetwarzanie dat

Rozbijamy datę na:
- rok (Year)
- miesiąc (Month)
- dzień (Day)

In [3]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2010,12,1
1,536365,71053,WHITE METAL LANTERN,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010,12,1
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2010,12,1
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010,12,1
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010,12,1


## 4. Dodatkowe miary

Aby umożliwić analizę przychodu, dodajemy kolumnę Revenue:

Revenue = Quantity * UnitPrice

In [7]:
df["Revenue"] = (df["Quantity"] * df["UnitPrice"]).round(2)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Year,Month,Day,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,2010,12,1,15.30
1,536365,71053,WHITE METAL LANTERN,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010,12,1,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,2010,12,1,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010,12,1,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,2010,12,1,20.34


## 5. Tabela faktów

Tworzymy tabelę faktów zawierającą kluczowe informacje o sprzedaży.

In [8]:
fact_sales = df[[
    "InvoiceNo",
    "StockCode",
    "CustomerID",
    "InvoiceDate",
    "Quantity",
    "UnitPrice",
    "Revenue"
]]

fact_sales.head()

,InvoiceNo,StockCode,CustomerID,InvoiceDate,Quantity,UnitPrice,Revenue
0,536365,85123A,17850.0,2010-12-01 08:26:00,6.0,2.55,15.30
1,536365,71053,17850.0,2010-12-01 08:26:00,6.0,3.39,20.34
2,536365,84406B,17850.0,2010-12-01 08:26:00,8.0,2.75,22.00
3,536365,84029G,17850.0,2010-12-01 08:26:00,6.0,3.39,20.34
4,536365,84029E,17850.0,2010-12-01 08:26:00,6.0,3.39,20.34


## 6. Load - zapis danych

Zapisujemy wynikową tabelę faktów do pliku CSV.

In [9]:
fact_sales.to_csv("fact_sales.csv", index=False)

## Podsumowanie

W ramach procesu ETL:
- wczytano dane (Extract)
- wyczyszczono i przygotowano dane (Transform)
- dodano miarę Revenue
- utworzono tabelę faktów
- zapisano dane do pliku CSV (Load)

Proces ETL umożliwia przygotowanie danych do dalszej analizy w hurtowni danych.